In [1]:
"""
Copyright (C) <2025>  <The Ohio State University>

This program is free software: you can redistribute it and/or modify it under
the terms of the GNU General Public License as published by the Free Software
Foundation, either version 3 of the License, or (at your option) any later version.
This program is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY;
without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.
See the GNU General Public License for more details. You should have received a copy of the
GNU General Public License along with this program.  If not, see <https://www.gnu.org/licenses/>

This notebook processes raw data from RNA folding simulations involving human
transcripts with natural indels. It tidies the data into a more accessible
format for downstream analysis.
""";

In [2]:
%reset -f
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import math
import os

In [4]:
working_directory = "/content/drive/MyDrive/work/data/RNA-protein/IndelPolymorphismsModulateDistalRNAProteinInteractions"
os.chdir(working_directory)

In [5]:
# --- Script Parameters ---
seq_length = 150
footprint = 7

In [6]:
# the data files that can be processed by this notebook
file_names = ['dataHuman/sameSites/',
              'dataHuman/badSites/']

# the dataset currently being processed
file_name = file_names[1]

# Input file
infile = open(file_name + "KdDDG.txt", "r")

# Output files
outfile1 = open(file_name + "all/tidiedKdDDG.txt", "w") #all computed HuR binding sites
outfile2 = open(file_name + "restricted/tidiedKdDDG.txt", "w") # only those that are lowest energy binding sites in entire sequence

In [7]:
# --- Write Headers to Output Files ---
header="distance \t log(kDRatio) \t DDG \t deltionSize \t bindSite \n"
outfile1.write(header)
outfile2.write(header)

56

In [8]:
# Define a tab character for formatting the output strings.
t='\t'

# Skip the header line of the input file.
infile.readline()

# --- Main Data Processing Loop ---
for line in infile:
    fields = line.split()

    # --- Extract Data from Columns ---
    Kd = fields[3]
    KdAfterDeletion = fields[10]
    BestinMotif = fields[5].split(":")[0]
    indelLoc = fields[13]
    deletionSize = fields[16]
    DDG = fields[19]
    bindingSite = fields[20] # Indicates if the binding site is "inMotif"

    # --- Calculate Derived Metrics ---
    # Calculate the ratio of binding affinities (Kd) before and after the deletion and apply a log transform scaled by RT
    kdRatio = float(KdAfterDeletion) / float(Kd)
    log_KD_Ratio = str(0.6163207755 * math.log(kdRatio))

    # --- Calculate Distance from Indel to Binding Site ---
    if int(BestinMotif) > int(indelLoc) + int(deletionSize):
        # Case 1: The binding site is downstream of the indel.
        distance = int(BestinMotif) - (int(indelLoc) + int(deletionSize)) - 1

    elif int(BestinMotif) < int(indelLoc) - footprint + 1:
        # Case 2: The binding site is upstream of the indel.
        distance =  int(indelLoc) - (footprint + int(BestinMotif))

    else:
        # Case 3: The indel overlaps with the binding site.
        distance = -1

    # --- Format and Write Output ---
    string = str(distance) + t + log_KD_Ratio + t + DDG + t + deletionSize + t + bindingSite + "\n"

    outfile1.write(string)

    # binding sites is the lowest energy in entire sequence
    if bindingSite == "inMotif":
        outfile2.write(string)



In [9]:
# Close all open file handles
outfile1.close()
outfile2.close()